Useful Links
- https://www.youtube.com/watch?v=VMj-3S1tku0
- https://github.com/karpathy/nn-zero-to-hero/tree/master



In [68]:
%pip install numpy matplotlib graphviz

17662.48s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


In [69]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
%matplotlib inline

In [ ]:
class Value:
  def __init__(self, data, _children= (), _op = '',label=''):
    self.data = data
    self._prev = set(_children)
    self._op = _op
    self.label = label
    self.grad = 0.0
    self._backward = lambda: None

  def __repr__(self) -> str:
    return f"Value(data={self.data})"

  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out

  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data**other, (self,), f'**{other}')
    def _backward():
      self.grad += (other * self.data**(other-1)) * out.grad
    out._backward = _backward
    return out

  def __rmul__(self, other):
    return self * other

  def __truediv__(self, other):
    return self * other**-1

  def __neg__(self):
    return self * -1

  def __sub__(self, other):
    return self + (-other)

  def tanh(self):
    t = math.tanh(self.data)
    out = Value(t, (self, ), 'tanh')
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out

  def exp(self):
    x = self.data
    out = Value(math.exp(x), (self, ), 'exp')
    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward
    return out

  def relu(self):
    out = Value(0 if self.data < 0 else self.data, (self, ), 'ReLU')
    def _backward():
      self.grad += (out.data > 0) * out.grad
    out._backward = _backward
    return out

  def sigmoid(self):
    x = self.data
    if x >= 0:
        s = 1 / (1 + math.exp(-x))
    else:
        exp_x = math.exp(x)
        s = exp_x / (1 + exp_x)
    out = Value(s, (self, ), 'sigmoid')
    def _backward():
      self.grad += out.data * (1 - out.data) * out.grad
    out._backward = _backward
    return out

  def trace(self):
    nodes, edges = set(), set()
    def build(value):
      if value not in nodes:
        nodes.add(value)
        for child in value._prev:
          edges.add((child, value))
          build(child)
    build(self)
    return nodes, edges

  def draw_dot(self):
    from graphviz import Digraph

    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})
    nodes, edges = self.trace()
    for node in nodes:
      uid = str(id(node))
      dot.node(name=uid, label="{ %s | data %.4f | grad %.4f }" % (node.label, node.data, node.grad), shape='record')
      if node._op:
        dot.node(name=uid + node._op, label=node._op)
        dot.edge(uid + node._op, uid)

    for child, parent in edges:
      dot.edge(str(id(child)), str(id(parent)) + parent._op)

    return dot

  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [ ]:
class Linear_Neuron:
  def __init__(self, nin):
    self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
    self.b = Value(0.0)

  def __call__(self, x):
    return sum((wi*xi for wi, xi in zip(self.w, x)), self.b)

  def parameters(self):
    return self.w + [self.b]

In [ ]:
class Tanh_Neuron:
  def __init__(self, nin):
    self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
    self.b = Value(0.0) # start biases at 0.0

  def __call__(self, x):
    act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
    return act.tanh()

  def parameters(self):
    return self.w + [self.b]

In [ ]:
class ReLU_Neuron:
  def __init__(self, nin):
    scale = math.sqrt(2.0 / nin) if nin > 0 else 1.0
    self.w = [Value(random.gauss(0, 1) * scale) for _ in range(nin)]
    self.b = Value(0.1)

  def __call__(self, x):
    act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
    return act.relu()

  def parameters(self):
    return self.w + [self.b]

In [ ]:
class Sigmoid_Neuron:
  def __init__(self, nin):
    self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
    self.b = Value(0.0)

  def __call__(self, x):
    act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
    return act.sigmoid()

  def parameters(self):
    return self.w + [self.b]

In [ ]:
class Layer:
  def __init__(self, neurons=None):
    self.neurons = []
    if neurons is not None:
      for neuron in neurons:
        self.add_neuron(neuron)

  def add_neuron(self, neuron):
    if self.neurons and len(neuron.w) != len(self.neurons[0].w):
      raise ValueError("All neurons in a layer must have the same input size")
    self.neurons.append(neuron)
    return self

  def __call__(self, x):
    if not self.neurons:
      raise ValueError("Cannot evaluate an empty layer")
    if isinstance(x, Value):
      x = [x]
    if len(x) != len(self.neurons[0].w):
      raise ValueError("Input size must match the layer's neuron input size")
    outs = [neuron(x) for neuron in self.neurons]
    return outs[0] if len(outs) == 1 else outs

  def parameters(self):
    return [parameter for neuron in self.neurons for parameter in neuron.parameters()]

In [ ]:
class MLP:
  def __init__(self, layers=None):
    self.layers = []
    if layers is not None:
      for layer in layers:
        self.add_layer(layer)

  def add_layer(self, layer):
    if not layer.neurons:
      raise ValueError("Cannot add an empty layer to an MLP")
    if self.layers and any(len(neuron.w) != len(self.layers[-1].neurons) for neuron in layer.neurons):
      raise ValueError("Layer input size must match the previous layer's output size")
    self.layers.append(layer)
    return self

  def __call__(self, x):
    if not self.layers:
      raise ValueError("Cannot evaluate an empty MLP")
    for layer in self.layers:
      x = layer(x)
    return x

  def parameters(self):
    return [parameter for layer in self.layers for parameter in layer.parameters()]

  def draw_dot(self):
    """Return a Graphviz diagram of inputs, activations, and fully connected layers."""
    from graphviz import Digraph

    if not self.layers or any(not layer.neurons for layer in self.layers):
      raise ValueError("Cannot visualize an empty MLP or layer")
    input_count = len(self.layers[0].neurons[0].w)
    activations = {"Tanh_Neuron": "tanh", "ReLU_Neuron": "ReLU",
                   "Sigmoid_Neuron": "sigmoid", "Linear_Neuron": "linear"}
    dot = Digraph(format='svg', graph_attr={
      'rankdir': 'LR', 'ranksep': '0.9', 'nodesep': '0.35',
      'splines': 'line', 'bgcolor': 'white', 'ordering': 'out',
    })
    dot.attr('node', shape='circle', style='filled', fixedsize='true',
             width='1.0', height='1.0', fontname='Helvetica', fontsize='11')
    dot.attr('edge', color='#9ca3af', arrowsize='0.6')

    previous_nodes = [f'input_{index}' for index in range(input_count)]
    with dot.subgraph(name='cluster_input') as column:
      column.attr(rank='same', label='Input', color='transparent', fontcolor='#dc2626')
      for index, node_id in enumerate(previous_nodes):
        column.node(node_id, label=f'x{index}', color='#dc2626', fillcolor='#fff5f5')

    for layer_index, layer in enumerate(self.layers):
      if any(len(neuron.w) != len(previous_nodes) for neuron in layer.neurons):
        raise ValueError("Layer input size must match the previous column's size")
      is_output = layer_index == len(self.layers) - 1
      heading = 'Output' if is_output else f'Hidden {layer_index + 1}'
      color, fill = ('#15803d', '#f0fdf4') if is_output else ('#2563eb', '#eff6ff')
      current_nodes = []
      with dot.subgraph(name=f'cluster_layer_{layer_index}') as column:
        column.attr(rank='same', label=heading, color='transparent', fontcolor=color)
        for neuron_index, neuron in enumerate(layer.neurons):
          node_id = f'layer_{layer_index}_neuron_{neuron_index}'
          activation = activations.get(type(neuron).__name__, type(neuron).__name__.removesuffix('_Neuron'))
          column.node(node_id, label=activation, color=color, fillcolor=fill)
          current_nodes.append(node_id)
      for source in previous_nodes:
        for target in current_nodes:
          dot.edge(source, target)
      previous_nodes = current_nodes

    return dot

  def predict(self, xs, threshold=0.0, verbose=False):
    """Predict scalar scores and binary classes without target labels."""
    xs = list(xs)
    if not xs:
      raise ValueError("Prediction requires a nonempty dataset")
    if not math.isfinite(threshold):
      raise ValueError("Threshold must be finite")

    predictions = []
    for point in xs:
      output = self(point)
      if not isinstance(output, Value) or not math.isfinite(output.data):
        raise ValueError("Binary prediction requires finite scalar model outputs")
      predictions.append(output.data)
    predicted_classes = [1.0 if prediction >= threshold else -1.0 for prediction in predictions]

    if verbose:
      print("Coordinate Predictions:")
      for index, (point, prediction, predicted_class) in enumerate(zip(xs, predictions, predicted_classes)):
        coordinates = ", ".join(f"{coordinate:.2f}" for coordinate in point)
        print(f"Point {index:2d} | Coordinates: ({coordinates}) | Score: {prediction:6.3f} | Predicted: {predicted_class:+.1f}")

    return {"predictions": predictions, "predicted_classes": predicted_classes}

  def evaluate(self, xs, ys, threshold=0.0, verbose=False):
    """Evaluate scalar binary predictions against labels -1 and +1."""
    xs, ys = list(xs), list(ys)
    if not xs or len(xs) != len(ys):
      raise ValueError("Evaluation requires a nonempty dataset with one label per point")
    if any(target not in (-1.0, 1.0) for target in ys):
      raise ValueError("Binary evaluation requires labels -1 and +1")

    result = self.predict(xs, threshold=threshold)
    predictions, predicted_classes = result["predictions"], result["predicted_classes"]
    correct = [prediction == target for prediction, target in zip(predicted_classes, ys)]
    accuracy = sum(correct) / len(correct)

    if verbose:
      print("Detailed Test Predictions:")
      for index, (point, target, prediction, is_correct) in enumerate(zip(xs, ys, predictions, correct)):
        coordinates = ", ".join(f"{coordinate:.2f}" for coordinate in point)
        result_label = "CORRECT" if is_correct else "WRONG"
        print(f"Point {index:2d} ({coordinates}) | Target: {target:4.1f} | Prediction: {prediction:6.3f} | [{result_label}]")
      print(f"Accuracy: {accuracy:.1%} ({sum(correct)}/{len(correct)})")

    return {**result, "correct": correct, "accuracy": accuracy}

  def train(self, xs, ys, epochs=1000, lr=0.01, alpha=0.001, verbose_step=100):
    """
    Trains using summed squared error and L2 weight decay (excluding biases).
    Each call starts with new random weights and constructor-default biases.
    To continue learning, use more epochs in a single call instead.
    """
    xs, ys = list(xs), list(ys)
    if not xs or len(xs) != len(ys):
      raise ValueError("Training requires a nonempty dataset with one label per point")
    for layer in self.layers:
      for neuron in layer.neurons:
        fresh_neuron = type(neuron)(len(neuron.w))
        neuron.w, neuron.b = fresh_neuron.w, fresh_neuron.b
    print(f"Training MLP model for {epochs} epochs with lr={lr}, alpha={alpha}...")

    for k in range(epochs + 1):
        # 1. Forward pass
        ypred = [self(x) for x in xs]

        # 2. Data loss (Mean Squared Error)
        data_loss = sum(((yout - ygt)**2 for ygt, yout in zip(ys, ypred)), Value(0.0))

        # 3. L2 weight decay penalty (excluding biases)
        weights_only = [p for layer in self.layers for neuron in layer.neurons for p in neuron.w]
        reg_loss = sum((w**2 for w in weights_only), Value(0.0)) * alpha

        # 4. Combined Loss
        loss = data_loss + reg_loss

        # 5. Backward pass (reset gradients first)
        for p in self.parameters():
            p.grad = 0.0
        loss.backward()

        # 6. Gradient update step
        for p in self.parameters():
            p.data += -lr * p.grad

        if k % verbose_step == 0:
            print(f"Step {k:4d} | Total Loss: {loss.data:.4f} | Data Loss: {data_loss.data:.4f} | Reg Loss: {reg_loss.data:.4f}")

    return loss

In [ ]:
class PatternDatasetGenerator:
    """
    A generic dataset generator to create random patterns in N dimensions
    for testing and training neural networks.
    """
    def __init__(self, random_seed=42):
        self.random_seed = random_seed
        random.seed(random_seed)
        np.random.seed(random_seed)

    def make_blobs(self, n_samples=150, n_dim=2, centers=2, noise=0.1):
        """
        Generates isotropic Gaussian blobs in N dimensions.
        """
        xs = []
        ys = []

        # Generate stable coordinates for cluster centers in N-D
        center_coords = []
        for c in range(centers):
            # Place centers randomly in hypercube space
            center_coords.append([random.uniform(-1.5, 1.5) for _ in range(n_dim)])

        samples_per_center = n_samples // centers
        for c, center in enumerate(center_coords):
            label = 1.0 if c % 2 == 0 else -1.0
            for _ in range(samples_per_center):
                point = [center[d] + random.gauss(0, noise) for d in range(n_dim)]
                xs.append(point)
                ys.append(label)
        return xs, ys

    def make_concentric_spheres(self, n_samples=150, n_dim=2, noise=0.05, r_inner=0.5, r_outer=1.2):
        """
        Generates concentric hyperspheres (circles in 2D, spheres in 3D) in N dimensions.
        """
        xs = []
        ys = []
        samples_per_class = n_samples // 2

        # Inner hypersphere (Class +1)
        for _ in range(samples_per_class):
            # Generate a random unit direction in N-D
            direction = np.random.normal(0, 1, n_dim)
            norm = np.linalg.norm(direction)
            if norm == 0:
                direction[0] = 1.0
                norm = 1.0
            unit_vector = direction / norm
            radius = r_inner + random.gauss(0, noise)
            xs.append((unit_vector * radius).tolist())
            ys.append(1.0)

        # Outer hypersphere (Class -1)
        for _ in range(samples_per_class):
            direction = np.random.normal(0, 1, n_dim)
            norm = np.linalg.norm(direction)
            if norm == 0:
                direction[0] = 1.0
                norm = 1.0
            unit_vector = direction / norm
            radius = r_outer + random.gauss(0, noise)
            xs.append((unit_vector * radius).tolist())
            ys.append(-1.0)

        return xs, ys

    def make_moons_nd(self, n_samples=150, n_dim=2, noise=0.1):
        """
        Generates interlocking moons in 2D, padded with random noise in higher dimensions if n_dim > 2.
        """
        xs = []
        ys = []
        samples_per_moon = n_samples // 2

        # Top Moon (Class +1)
        for i in range(samples_per_moon):
            theta = math.pi * (i / (samples_per_moon - 1))
            x0 = math.cos(theta) + random.gauss(0, noise)
            x1 = math.sin(theta) + random.gauss(0, noise)
            extra_dims = [random.gauss(0, noise) for _ in range(n_dim - 2)]
            xs.append([x0, x1] + extra_dims)
            ys.append(1.0)

        # Bottom Moon (Class -1)
        for i in range(samples_per_moon):
            theta = math.pi * (i / (samples_per_moon - 1))
            x0 = 1.0 - math.cos(theta) + random.gauss(0, noise)
            x1 = 0.5 - math.sin(theta) + random.gauss(0, noise)
            extra_dims = [random.gauss(0, noise) for _ in range(n_dim - 2)]
            xs.append([x0, x1] + extra_dims)
            ys.append(-1.0)

        return xs, ys

    @staticmethod
    def plot_dataset(xs, ys, title="Dataset Visualization"):
        """
        Plots the dataset if the dimensions of the data are 2D or 3D.
        """
        xs = np.array(xs)
        ys = np.array(ys)
        n_dim = xs.shape[1]

        if n_dim != 2 and n_dim != 3:
            print(f"Plotting is only supported for 2D or 3D datasets. The data has {n_dim} dimensions.")
            return

        fig = plt.figure(figsize=(9, 7))
        pos_mask = (ys == 1.0)
        neg_mask = (ys == -1.0)

        if n_dim == 2:
            plt.scatter(xs[pos_mask, 0], xs[pos_mask, 1], color='blue', marker='o', label='Class +1', edgecolors='k', alpha=0.8)
            plt.scatter(xs[neg_mask, 0], xs[neg_mask, 1], color='red', marker='x', label='Class -1', alpha=0.8)
            plt.xlabel('Dimension 0')
            plt.ylabel('Dimension 1')
            plt.grid(True, linestyle='--', alpha=0.5)
        elif n_dim == 3:
            ax = fig.add_subplot(111, projection='3d')
            ax.scatter(xs[pos_mask, 0], xs[pos_mask, 1], xs[pos_mask, 2], color='blue', marker='o', label='Class +1', edgecolors='k', alpha=0.8)
            ax.scatter(xs[neg_mask, 0], xs[neg_mask, 1], xs[neg_mask, 2], color='red', marker='x', label='Class -1', alpha=0.8)
            ax.set_xlabel('Dimension 0')
            ax.set_ylabel('Dimension 1')
            ax.set_zlabel('Dimension 2')

        plt.title(title)
        plt.legend()
        plt.show()

    @staticmethod
    def plot_dataset_with_overlay(xs, ys, overlay_xs=None, overlay_ys=None,
                                  title="Dataset Visualization with Overlay",
                                  base_alpha=0.3, overlay_marker='*', overlay_size=200):
        """
        Plots a base dataset and optionally overlays another set of points with custom styling.
        """
        xs = np.array(xs)
        ys = np.array(ys)
        n_dim = xs.shape[1]

        if n_dim != 2:
            print("Overlay plotting is currently only supported for 2D datasets.")
            return

        plt.figure(figsize=(10, 8))
        pos_mask = (ys == 1.0)
        neg_mask = (ys == -1.0)

        # Plot base training dataset
        plt.scatter(xs[pos_mask, 0], xs[pos_mask, 1], color='blue', marker='o', alpha=base_alpha, label='Base Class +1', edgecolors='k')
        plt.scatter(xs[neg_mask, 0], xs[neg_mask, 1], color='red', marker='x', alpha=base_alpha, label='Base Class -1')

        # Plot overlay dataset (e.g. random validation points or test points)
        if overlay_xs is not None and overlay_ys is not None:
            overlay_xs = np.array(overlay_xs)
            overlay_ys = np.array(overlay_ys)

            overlay_pos_mask = (overlay_ys == 1.0)
            overlay_neg_mask = (overlay_ys == -1.0)

            if np.any(overlay_pos_mask):
                plt.scatter(overlay_xs[overlay_pos_mask, 0], overlay_xs[overlay_pos_mask, 1],
                            color='blue', marker=overlay_marker, s=overlay_size,
                            edgecolors='black', linewidths=1.5, label='Overlay Class +1')
            if np.any(overlay_neg_mask):
                plt.scatter(overlay_xs[overlay_neg_mask, 0], overlay_xs[overlay_neg_mask, 1],
                            color='red', marker=overlay_marker, s=overlay_size,
                            edgecolors='black', linewidths=1.5, label='Overlay Class -1')

        plt.xlabel('Dimension 0')
        plt.ylabel('Dimension 1')
        plt.title(title)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.legend(loc='upper right')
        plt.show()

    @staticmethod
    def plot_decision_boundary(model, xs, ys, title="Learned Decision Boundary",
                               x_limits=None, y_limits=None, resolution=100, threshold=0.0,
                               overlay_xs=None, overlay_ys=None, overlay_predictions=None,
                               probe_xs=None, probe_predictions=None, overlay_markers=('o', 'x')):
        """Plot a 2D boundary, labeled correctness rings, and unlabeled prediction stars."""
        points = np.asarray(xs, dtype=float)
        labels = np.asarray(ys)
        if points.ndim != 2 or points.shape[1] != 2 or len(points) == 0:
            raise ValueError("Decision boundaries require a nonempty 2D dataset")
        if labels.ndim != 1 or len(labels) != len(points):
            raise ValueError("Each point must have one label")
        if not np.isfinite(points).all():
            raise ValueError("Dataset coordinates must be finite")
        if not isinstance(resolution, int) or resolution < 2:
            raise ValueError("Resolution must be an integer of at least 2")
        if not math.isfinite(threshold):
            raise ValueError("Threshold must be finite")
        if len(overlay_markers) != 2:
            raise ValueError("Supply one overlay marker for each binary class")

        has_overlay = any(value is not None for value in (overlay_xs, overlay_ys, overlay_predictions))
        bounds_points = points
        if has_overlay:
            if any(value is None for value in (overlay_xs, overlay_ys, overlay_predictions)):
                raise ValueError("Overlay points, labels, and predictions must be supplied together")
            overlay_points = np.asarray(overlay_xs, dtype=float)
            overlay_labels = np.asarray(overlay_ys)
            predictions = np.asarray(overlay_predictions, dtype=float)
            if overlay_points.ndim != 2 or overlay_points.shape[1] != 2 or len(overlay_points) == 0:
                raise ValueError("Overlay requires a nonempty 2D dataset")
            if overlay_labels.shape != (len(overlay_points),) or predictions.shape != (len(overlay_points),):
                raise ValueError("Each overlay point must have one label and one scalar prediction")
            if not np.isfinite(overlay_points).all() or not np.isfinite(predictions).all():
                raise ValueError("Overlay coordinates and predictions must be finite")
            if not np.isin(overlay_labels, (-1.0, 1.0)).all():
                raise ValueError("Overlay requires labels -1 and +1")
            correct = np.where(predictions >= threshold, 1.0, -1.0) == overlay_labels
            bounds_points = np.concatenate((points, overlay_points))

        has_probes = probe_xs is not None or probe_predictions is not None
        if has_probes:
            if probe_xs is None or probe_predictions is None:
                raise ValueError("Probe points and predictions must be supplied together")
            probe_points = np.asarray(probe_xs, dtype=float)
            probe_scores = np.asarray(probe_predictions, dtype=float)
            if probe_points.ndim != 2 or probe_points.shape[1] != 2 or len(probe_points) == 0:
                raise ValueError("Probes require a nonempty 2D dataset")
            if probe_scores.shape != (len(probe_points),):
                raise ValueError("Each probe point must have one scalar prediction")
            if not np.isfinite(probe_points).all() or not np.isfinite(probe_scores).all():
                raise ValueError("Probe coordinates and predictions must be finite")
            probe_classes = np.where(probe_scores >= threshold, 1.0, -1.0)
            bounds_points = np.concatenate((bounds_points, probe_points))

        padding = np.maximum(np.ptp(bounds_points, axis=0) * 0.1, 0.1)
        if x_limits is None:
            x_limits = (bounds_points[:, 0].min() - padding[0], bounds_points[:, 0].max() + padding[0])
        if y_limits is None:
            y_limits = (bounds_points[:, 1].min() - padding[1], bounds_points[:, 1].max() + padding[1])
        for limits in (x_limits, y_limits):
            if len(limits) != 2 or not np.isfinite(limits).all() or limits[0] >= limits[1]:
                raise ValueError("Plot limits must contain two increasing finite values")

        x0_grid, x1_grid = np.meshgrid(
            np.linspace(*x_limits, resolution), np.linspace(*y_limits, resolution)
        )
        outputs = np.array([
            model([float(x0), float(x1)]).data
            for x0, x1 in zip(x0_grid.ravel(), x1_grid.ravel())
        ]).reshape(x0_grid.shape)

        figure, axis = plt.subplots(figsize=(10, 8))
        shading = axis.contourf(x0_grid, x1_grid, outputs, levels=50, cmap='RdYlBu', alpha=0.4 if has_overlay or has_probes else 0.6)
        if outputs.min() < threshold < outputs.max():
            axis.contour(x0_grid, x1_grid, outputs, levels=[threshold], colors='black', linewidths=1)
        for index, (target, color, marker) in enumerate(((1.0, 'blue', 'o'), (-1.0, 'red', 'x'))):
            selected = points[labels == target]
            if has_overlay or has_probes:
                axis.scatter(selected[:, 0], selected[:, 1], color=color, marker=marker,
                             alpha=0.15, s=20, label=f'Train Class {target:+.0f}')
            else:
                axis.scatter(selected[:, 0], selected[:, 1], color=color, marker=marker, label=f'Class {target:+.0f}')
            if has_overlay:
                selected = overlay_points[overlay_labels == target]
                axis.scatter(selected[:, 0], selected[:, 1], color=color, marker=overlay_markers[index],
                             s=120, label=f'Test Class {target:+.0f}')
            if has_probes:
                selected = probe_points[probe_classes == target]
                if len(selected):
                    axis.scatter(selected[:, 0], selected[:, 1], color=color, marker='*', s=220,
                                 edgecolors='black', linewidths=1.5, label=f'Probe Prediction {target:+.0f}')
        if has_overlay:
            for is_correct, color, label in ((True, 'limegreen', 'Correct Test Point'), (False, 'darkorange', 'Wrong Test Point')):
                selected = overlay_points[correct == is_correct]
                if len(selected):
                    axis.scatter(selected[:, 0], selected[:, 1], marker='o', s=240,
                                 facecolors='none', edgecolors=color, linewidths=2.5, label=label)
        if has_overlay or has_probes:
            axis.grid(True, linestyle='--', alpha=0.3)

        axis.set_title(title)
        axis.set_xlabel('X0')
        axis.set_ylabel('X1')
        figure.colorbar(shading, ax=axis, label='Network output')
        axis.legend()
        plt.show()
        return figure, axis

    @staticmethod
    def train_test_split(xs, ys, train_size, random_seed=42):
        """Shuffle paired samples into nonempty train/test lists without changing the global RNG."""
        xs, ys = list(xs), list(ys)
        if len(xs) != len(ys):
            raise ValueError("Each point must have one label")
        if isinstance(train_size, bool) or not isinstance(train_size, int) or not 0 < train_size < len(xs):
            raise ValueError("Train size must be an integer leaving nonempty train and test sets")

        paired_dataset = list(zip(xs, ys))
        random.Random(random_seed).shuffle(paired_dataset)
        xs_shuffled, ys_shuffled = zip(*paired_dataset)
        xs_shuffled, ys_shuffled = list(xs_shuffled), list(ys_shuffled)
        return (xs_shuffled[:train_size], ys_shuffled[:train_size],
                xs_shuffled[train_size:], ys_shuffled[train_size:])

    @staticmethod
    def make_random_points(n_samples, bounds, random_seed=42):
        """Generate unlabeled uniform points within one (lower, upper) pair per dimension."""
        if isinstance(n_samples, bool) or not isinstance(n_samples, int) or n_samples < 1:
            raise ValueError("Sample count must be a positive integer")
        bounds = np.asarray(bounds, dtype=float)
        if bounds.ndim != 2 or bounds.shape[1] != 2 or len(bounds) == 0:
            raise ValueError("Bounds must contain a lower and upper limit for each dimension")
        if not np.isfinite(bounds).all() or np.any(bounds[:, 0] >= bounds[:, 1]):
            raise ValueError("Bounds must contain increasing finite limits")
        rng = random.Random(random_seed)
        return [[rng.uniform(lower, upper) for lower, upper in bounds] for _ in range(n_samples)]

In [ ]:
n = (
  MLP()
  .add_layer(Layer([Tanh_Neuron(3) for _ in range(4)]))
  .add_layer(Layer([Tanh_Neuron(4) for _ in range(4)]))
  .add_layer(Layer().add_neuron(Tanh_Neuron(4)))
)

xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

for k in range(1000):

  # forward pass
  ypred = [n(x) for x in xs]
  # By adding Value(0.0) as the start parameter, sum() knows how to accumulate custom Value instances
  loss = sum(((yout - ygt)**2 for ygt, yout in zip(ys, ypred)), Value(0.0))

  # backward pass
  for p in n.parameters():
    p.grad = 0.0
  loss.backward()

  # update
  for p in n.parameters():
    p.data += -0.01 * p.grad

  print(k, loss.data)

ypred

In [ ]:
relu_n = MLP([
  Layer([ReLU_Neuron(3) for _ in range(4)]),
  Layer([ReLU_Neuron(4) for _ in range(4)]),
  Layer([Linear_Neuron(4)]),
])

print("Training MLP using ReLU Activation:")
for k in range(1000):
  # forward pass
  ypred = [relu_n(x) for x in xs]
  loss = sum(((yout - ygt)**2 for ygt, yout in zip(ys, ypred)), Value(0.0))

  # backward pass
  for p in relu_n.parameters():
    p.grad = 0.0
  loss.backward()

  # update (using a stable learning rate)
  for p in relu_n.parameters():
    p.data += -0.01 * p.grad

  if k % 10 == 0:
    print(f"Step {k:3d} | Loss: {loss.data:.4f}")

In [ ]:
ypred

In [ ]:
sigmoid_n = MLP([
  Layer([Sigmoid_Neuron(3) for _ in range(4)]),
  Layer([Sigmoid_Neuron(4) for _ in range(4)]),
  Layer([Sigmoid_Neuron(4)]),
])

print("Training MLP using Sigmoid Activation:")
for k in range(1000):
  # forward pass
  ypred = [sigmoid_n(x) for x in xs]
  loss = sum(((yout - ygt)**2 for ygt, yout in zip(ys, ypred)), Value(0.0))

  # backward pass
  for p in sigmoid_n.parameters():
    p.grad = 0.0
  loss.backward()

  # update (using a stable learning rate)
  for p in sigmoid_n.parameters():
    p.data += -0.01 * p.grad

  if k % 10 == 0:
    print(f"Step {k:3d} | Loss: {loss.data:.4f}")

In [ ]:
ypred

In [ ]:
generator = PatternDatasetGenerator(random_seed=42)
xs_moons, ys_moons = generator.make_moons_nd(
    n_samples=120, n_dim=2, noise=0.1
)

# Split into Train and Test
xs_moons_train = xs_moons[:100]
ys_moons_train = ys_moons[:100]
xs_moons_test = xs_moons[100:]
ys_moons_test = ys_moons[100:]

print(f"Generated {len(xs_moons_train)} training and {len(xs_moons_test)} testing 2D Moon samples.")

In [ ]:
generator.plot_dataset(
    xs_moons, ys_moons,
    title="2D Interleaving Moons Dataset"
)

In [ ]:
# Overwrite our training and testing inputs to use the new 2D moons dataset
xs_train = xs_moons_train
ys_train = ys_moons_train
xs_test = xs_moons_test
ys_test = ys_moons_test

print("Successfully reassigned the global variables:")
print(f"xs_train now has {len(xs_train)} samples of 2D coordinates (e.g., {xs_train[0]})")
print(f"ys_train now contains their corresponding binary labels (e.g., {ys_train[0]})")

In [ ]:
random.seed(42)
nn = MLP([
  Layer([ReLU_Neuron(2) for _ in range(4)]),
  Layer([ReLU_Neuron(4) for _ in range(4)]),
  Layer([Linear_Neuron(4)]),
])

learning_rate = 0.0005
loss = nn.train(
    xs_train, ys_train, epochs=1000,
    lr=learning_rate, alpha=0.1, verbose_step=100
)

In [ ]:
boundary_figure, boundary_axis = PatternDatasetGenerator.plot_decision_boundary(
    nn, xs_train, ys_train,
    title="Learned Decision Boundary on 2D Moons (ReLU MLP)",
    x_limits=(-1.5, 2.5), y_limits=(-1.0, 1.5)
)

In [ ]:
test_results = nn.evaluate(xs_test, ys_test, verbose=True)

test_figure, test_axis = PatternDatasetGenerator.plot_decision_boundary(
    nn, xs_train, ys_train,
    title="Test Set Predictions on Decision Boundary (ReLU MLP)",
    x_limits=(-1.5, 2.5), y_limits=(-1.0, 1.5),
    overlay_xs=xs_test, overlay_ys=ys_test,
    overlay_predictions=test_results["predictions"]
)

### Shuffling and Re-Splitting the Moons Dataset
Let's combine our coordinate points (`xs_moons`) and labels (`ys_moons`), shuffle them randomly to ensure balanced regional coverage, and then split them into train and test sets.

In [ ]:
xs_train, ys_train, xs_test, ys_test = PatternDatasetGenerator.train_test_split(
    xs_moons, ys_moons, train_size=100, random_seed=42
)

print(f"Shuffled training set size: {len(xs_train)}")
print(f"Shuffled test set size: {len(xs_test)}")

PatternDatasetGenerator.plot_dataset(
    xs_train, ys_train, title="Shuffled Moons Training Set"
)

### 🏋️‍♂️ Training the Tanh MLP on the Shuffled and Balanced Dataset
We will initialize a fresh MLP from three explicit tanh layers with 4, 4, and 1 neurons, apply Xavier scaling to its weights, and train it using our stable learning rate and L2 regularization.

In [ ]:
tanh_nn = MLP([
  Layer([Tanh_Neuron(2) for _ in range(4)]),
  Layer([Tanh_Neuron(4) for _ in range(4)]),
  Layer([Tanh_Neuron(4)]),
])

test_final_loss = tanh_nn.train(xs_train, ys_train, epochs=1000, lr=0.01, alpha=0.001, verbose_step=100)

In [ ]:
structured_results = tanh_nn.evaluate(xs_test, ys_test)
print(f"Structured Test Set Accuracy: {structured_results['accuracy']:.1%}")

new_test_xs = PatternDatasetGenerator.make_random_points(
    n_samples=15, bounds=[(-1.2, 2.2), (-0.8, 1.2)], random_seed=101
)
random_results = tanh_nn.predict(new_test_xs, verbose=True)

merged_figure, merged_axis = PatternDatasetGenerator.plot_decision_boundary(
    tanh_nn, xs_train, ys_train,
    title="Merged Evaluation: Structured Test Set & Unseen Random Coordinate Items",
    x_limits=(-1.5, 2.5), y_limits=(-1.0, 1.5),
    overlay_xs=xs_test, overlay_ys=ys_test,
    overlay_predictions=structured_results["predictions"], overlay_markers=('o', 's'),
    probe_xs=new_test_xs, probe_predictions=random_results["predictions"]
)

In [ ]:
# Demo the dataset generator
generator = PatternDatasetGenerator(random_seed=42)

# 1. Generate concentric circles in 2D
xs_2d, ys_2d = generator.make_concentric_spheres(n_samples=160, n_dim=2, noise=0.08)
generator.plot_dataset(xs_2d, ys_2d, title="2D Concentric Circles")

# 2. Intantiate NN
tanh_nn = MLP([
  Layer([Tanh_Neuron(2) for _ in range(4)]),
  Layer([Tanh_Neuron(4) for _ in range(4)]),
  Layer([Tanh_Neuron(4)]),
])

# 3. Train it
tanh_nn.train(xs_2d, ys_2d, epochs=1000, lr=0.01, alpha=0.001, verbose_step=100)


In [ ]:
random_points = PatternDatasetGenerator.make_random_points(
    n_samples=50, bounds=[(-1.5, 1.5), (-1.5, 1.5)], random_seed=42
)
predicted_labels = tanh_nn.predict(random_points)["predicted_classes"]


PatternDatasetGenerator.plot_dataset_with_overlay(
    xs=xs_2d,
    ys=ys_2d,
    overlay_xs=random_points,
    overlay_ys=predicted_labels,
    title="Predicted Random Points on 2D Concentric Circles",
    base_alpha=0.3,
    overlay_marker='*',
    overlay_size=220
)

In [ ]:
# Generate a test
xs_blobs, ys_blobs = generator.make_concentric_spheres(
    n_samples=200, n_dim=2, noise=0.2
)

print(f"Generated {len(xs_blobs)} points")
PatternDatasetGenerator.plot_dataset(xs_blobs, ys_blobs, title="2D Gaussian Blobs")



In [ ]:
blobs_nn = MLP([
    Layer([Sigmoid_Neuron(2) for _ in range(3)]),
    Layer([Sigmoid_Neuron(3) for _ in range(3)]),
    Layer([Sigmoid_Neuron(3)]),
])

print(f"Layer widths: {[len(layer.neurons) for layer in blobs_nn.layers]}")
print(f"Parameters: {len(blobs_nn.parameters())}")
blobs_nn.draw_dot()

In [ ]:
blobs_nn.train(xs_blobs, ys_blobs, 1000, 0.01, 0.005)

In [ ]:
blob_bounds = [
    (min(point[dimension] for point in xs_blobs) - 0.2,
     max(point[dimension] for point in xs_blobs) + 0.2)
    for dimension in range(2)
]
random_blob_xs = PatternDatasetGenerator.make_random_points(
    n_samples=50, bounds=blob_bounds, random_seed=101
)

# Sigmoid scores use 0.5; signed activation scores use 0.0.
blob_threshold = 0.5 if isinstance(blobs_nn.layers[-1].neurons[0], Sigmoid_Neuron) else 0.0
random_blob_results = blobs_nn.predict(random_blob_xs, threshold=blob_threshold)

PatternDatasetGenerator.plot_dataset_with_overlay(
    xs=xs_blobs, ys=ys_blobs,
    overlay_xs=random_blob_xs,
    overlay_ys=random_blob_results["predicted_classes"],
    title="Predicted Random Points on 2D Gaussian Blobs",
    base_alpha=0.3, overlay_marker='*', overlay_size=220
)
